# Interactive Line Editor with Bokeh

- Drag points to update the line
- Double-click to add a new point

In [4]:
import numpy as np
from bokeh.plotting import figure, show, output_notebook
from bokeh.models import ColumnDataSource, PointDrawTool, CustomJS, Button
from bokeh.layouts import column
from bokeh.events import DoubleTap

output_notebook()

# 1. Data Source for the Blue Points
points_source = ColumnDataSource(data=dict(x=[1, 2, 3, 4], y=[2, 3, 5, 4]))

# 2. Data Source for the Line & its draggable Red Handles
# We use two points to define the line: [start_point, end_point]
line_data = ColumnDataSource(data=dict(x=[-1, 6], y=[1, 6]))

# 3. Create the Figure
p = figure(
    title="Manual & Auto Regression: Drag Blue points (Auto) or Red handles (Manual)",
    tools="pan,wheel_zoom,reset",
    width=700, height=500,
    x_range=[-2, 10], y_range=[-2, 10]
)

# Renderers
p.line('x', 'y', source=line_data, line_width=4, color='#FB8072', line_alpha=0.6)
line_handles = p.circle('x', 'y', source=line_data, size=15, color='red', legend_label="Line Handles")
points_glyph = p.circle('x', 'y', source=points_source, size=12, color='#1F77B4', legend_label="Data Points")

# 4. JavaScript for Automatic Regression (Triggered by Blue Points)
auto_regression_js = """
    const px = points_source.data['x'];
    const py = points_source.data['y'];
    const n = px.length;
    if (n < 2) return;

    let sumX = 0, sumY = 0, sumXY = 0, sumXX = 0;
    for (let i = 0; i < n; i++) {
        sumX += px[i]; sumY += py[i];
        sumXY += px[i] * py[i]; sumXX += px[i] * px[i];
    }

    const slope = (n * sumXY - sumX * sumY) / (n * sumXX - sumX * sumX);
    const intercept = (sumY - slope * sumX) / n;

    // Update the red handles to match the new math
    const x_min = -2; const x_max = 10;
    line_data.data['x'] = [x_min, x_max];
    line_data.data['y'] = [slope * x_min + intercept, slope * x_max + intercept];
    line_data.change.emit();
"""

# 5. JavaScript for Manual Dragging (Triggered by Red Handles)
# When you drag the line, we don't need math, Bokeh just moves the coordinates.
# We just need to make sure the line draws between the two handles.

# Link Callbacks
calc_callback = CustomJS(args=dict(points_source=points_source, line_data=line_data), code=auto_regression_js)
points_source.js_on_change('data', calc_callback)

# 6. Setup Tools
# PointDrawTool for the Blue Points
draw_points = PointDrawTool(renderers=[points_glyph])
# PointDrawTool for the Red Line Handles
draw_line = PointDrawTool(renderers=[line_handles])

p.add_tools(draw_points, draw_line)
p.toolbar.active_tap = draw_points # Default to adding/moving blue points

# 7. Add logic to reset to "Best Fit" on double tap
reset_js = """
    points_source.change.emit(); // This triggers the auto_regression_js
"""
p.js_on_event(DoubleTap, CustomJS(args=dict(points_source=points_source), code=reset_js))

# Initial fit
p.js_on_event('mouseenter', calc_callback)

show(p)

Loading BokehJS ...